# Notebook 06: Shadow Mode Runner
## Run real agency notes through NB01 → NB02 → NB03

Takes raw inputs from `data/shadow/inputs/`, runs the full pipeline,
and saves system output + creates evaluation templates.

In [ ]:
import json
import os
import glob
import sys
from dataclasses import asdict
from datetime import datetime

sys.path.insert(0, os.path.abspath(os.getcwd()))

BASE = os.path.abspath(os.getcwd())
SHADOW_DIR = os.path.join(BASE, '..', 'data', 'shadow')
INPUTS_DIR = os.path.join(SHADOW_DIR, 'inputs')
OUTPUTS_DIR = os.path.join(SHADOW_DIR, 'outputs')
EVALS_DIR = os.path.join(SHADOW_DIR, 'evaluations')

print(f"Shadow mode directory: {SHADOW_DIR}")
print(f"Inputs: {INPUTS_DIR}")
print(f"Outputs: {OUTPUTS_DIR}")
print(f"Evaluations: {EVALS_DIR}")

In [ ]:
# Load NB01/NB02/NB03 models
def load_nb(path):
    with open(path, 'r') as f:
        nb = json.load(f)
    ns = {}
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            exec(cell['source'], ns)
    return ns

nb02 = load_nb('notebooks/02_gap_and_decision.ipynb')
nb03 = load_nb('notebooks/03_session_strategy.ipynb')

run_gap_and_decision = nb02['run_gap_and_decision']
build_session_output = nb03['build_session_output']
CanonicalPacket = nb02['CanonicalPacket']
Slot = nb02['Slot']
EvidenceRef = nb02['EvidenceRef']
UnknownField = nb02['UnknownField']

print("Models loaded.")

In [ ]:
# Helper: Build CanonicalPacket from raw fixture format
def build_packet_from_shadow(data):
    """
    For shadow mode, we simulate NB01 by extracting what we can from raw_input.
    In the real system, NB01 would do LLM extraction.
    For now, we use the same pattern as raw_fixtures.py: if there's an expected block,
    use it. Otherwise, build a minimal packet from raw text metadata.
    """
    facts = {}
    expected = data.get("expected")
    if expected and expected.get("extracted_fields"):
        for fname, info in expected["extracted_fields"].items():
            facts[fname] = Slot(
                value=info.get("value"),
                confidence=info.get("confidence", 0.8),
                authority_level=info.get("authority", "explicit_owner"),
            )
    else:
        # No expected block — build minimal packet from raw_input
        # In real shadow mode, NB01 would do actual extraction
        # For now, we flag that extraction is needed
        facts["raw_text_length"] = Slot(value=len(data.get("raw_input", "")),
                                         confidence=1.0, authority_level="explicit_owner")

    unknowns = []
    if expected:
        for uf in expected.get("expected_unknowns", []):
            unknowns.append(UnknownField(field_name=uf, reason="not_extracted_yet"))

    contradictions = []
    if expected:
        for ctype in expected.get("expected_contradictions", []):
            field_map = {
                "date_conflict": "travel_dates",
                "destination_conflict": "destination_city",
                "budget_conflict": "budget_range",
                "traveler_count_conflict": "traveler_count",
            }
            fname = field_map.get(ctype, ctype)
            contradictions.append({"field_name": fname, "values": ["value_a", "value_b"], "sources": ["shadow"]})

    return CanonicalPacket(
        packet_id=data["id"],
        created_at=datetime.now().isoformat(),
        last_updated=datetime.now().isoformat(),
        facts=facts,
        unknowns=unknowns,
        contradictions=contradictions,
        stage="discovery",
    )

In [ ]:
# Run pipeline on one shadow input
def run_shadow_case(filepath):
    """
    Load a shadow input, run NB01→NB02→NB03, save output + eval template.
    """
    with open(filepath, 'r') as f:
        data = json.load(f)

    shadow_id = data["id"]
    raw_input = data.get("raw_input", "")

    if not raw_input:
        print(f"  SKIP: {shadow_id} — raw_input is empty. Paste a real note and re-run.")
        return None

    print(f"\n{'='*60}")
    print(f"  {shadow_id}")
    print(f"{'='*60}")
    print(f"  Source: {data.get('source_type', 'unknown')}")
    print(f"  Input: {raw_input[:100]}{'...' if len(raw_input) > 100 else ''}")

    # NB01: Build packet
    packet = build_packet_from_shadow(data)

    # NB02: Decision
    try:
        decision = run_gap_and_decision(packet)
    except Exception as e:
        print(f"  ERROR in NB02: {e}")
        return None

    # NB03: Session strategy
    try:
        output = build_session_output(decision, packet)
    except Exception as e:
        print(f"  ERROR in NB03: {e}")
        return None

    # Build system output
    system_output = {
        "nb01_packet": {
            "packet_id": packet.packet_id,
            "facts": {k: {"value": v.value, "authority": v.authority_level, "confidence": v.confidence}
                      for k, v in packet.facts.items()},
            "derived_signals": {k: {"value": v.value, "authority": v.authority_level, "confidence": v.confidence}
                               for k, v in packet.derived_signals.items()},
            "hypotheses": {k: {"value": v.value, "authority": v.authority_level, "confidence": v.confidence}
                          for k, v in packet.hypotheses.items()},
            "unknowns": [asdict(u) for u in packet.unknowns],
            "contradictions": packet.contradictions,
        },
        "nb02_decision": {
            "decision_state": decision.decision_state,
            "hard_blockers": decision.hard_blockers,
            "soft_blockers": decision.soft_blockers,
            "confidence": decision.confidence_score,
            "follow_up_questions": decision.follow_up_questions,
            "rationale": decision.rationale,
        },
        "nb03_strategy": {
            "session_goal": output.strategy.session_goal,
            "priority_sequence": output.strategy.priority_sequence,
            "tonal_guardrails": output.strategy.tonal_guardrails,
            "suggested_opening": output.strategy.suggested_opening,
            "risk_flags": output.strategy.risk_flags,
            "exit_criteria": output.strategy.exit_criteria,
            "next_action": output.strategy.next_action,
            "prompts": {
                "system_context": output.prompts.system_context,
                "user_message": output.prompts.user_message,
                "constraints": output.prompts.constraints,
                "internal_notes": output.prompts.internal_notes,
            }
        }
    }

    # Save system output
    output_path = os.path.join(OUTPUTS_DIR, f"{shadow_id}_output.json")
    with open(output_path, 'w') as f:
        json.dump(system_output, f, indent=2)
    print(f"  Decision: {decision.decision_state}")
    print(f"  Confidence: {decision.confidence_score:.0%}")
    print(f"  Hard blockers: {decision.hard_blockers or 'None'}")
    print(f"  Output saved: {output_path}")

    # Create evaluation template
    eval_template = {
        "id": shadow_id,
        "verdict": None,
        "nb01_issues": [],
        "nb02_issues": [],
        "nb03_issues": [],
        "overall_issue": "",
        "what_human_would_do": "",
        "severity": None,
        "reviewer_notes": "",
    }
    eval_path = os.path.join(EVALS_DIR, f"{shadow_id}_eval.json")
    with open(eval_path, 'w') as f:
        json.dump(eval_template, f, indent=2)
    print(f"  Eval template saved: {eval_path}")

    return system_output

In [ ]:
# Run all shadow inputs
print("\n" + "=" * 60)
print("  SHADOW MODE RUNNER")
print("=" * 60)

input_files = sorted(glob.glob(os.path.join(INPUTS_DIR, "shadow_*.json")))
print(f"\nFound {len(input_files)} shadow input files.")

results = []
for filepath in input_files:
    result = run_shadow_case(filepath)
    if result:
        results.append(result)

print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
print(f"\n  Total inputs: {len(input_files)}")
print(f"  With raw_input: {len(results)}")
print(f"  Empty (waiting for notes): {len(input_files) - len(results)}")
print(f"\n  To add real notes:")
print(f"    1. Edit data/shadow/inputs/shadow_XXX.json")
print(f"    2. Paste raw agency note into 'raw_input'")
print(f"    3. Set 'source_type' (whatsapp/call_notes/crm/email)")
print(f"    4. Re-run this notebook")